## Модели KNN и SVM

In [1]:
import pandas as pd

# считываем датасеты с обработанным текстом и числовыми показателями
data = pd.read_csv('../EDA/key_inf_dollar.csv', parse_dates=['date'])
text = pd.read_json('../EDA/cbr_key-rate_press_releases_processed.json')

In [2]:
text.dropna(subset=['decision'], inplace=True)
text.reset_index(drop=True, inplace=True)

Из текстовых данных необходимо взять: количество слов, решение по ставке и дату, чтобы сопоставить с числовыми данными

In [3]:
import dateparser

# переводим дату в формат дат
text["date"] = text["date"].apply(lambda x: dateparser.parse(x))

/var/folders/lp/20xw8zx12gq7xct54lgfpkyw0000gn/T/ipykernel_23879/1769912476.py:4: DeprecationWarning: Parsing dates involving a day of month without a year specified is ambiguious
and fails to parse leap day. The default behavior will change in Python 3.15
to either always raise an exception or to use a different default year (TBD).
To avoid trouble, add a specific year to the input & format.
See https://github.com/python/cpython/issues/70647.
  text["date"] = text["date"].apply(lambda x: dateparser.parse(x))


In [10]:
#добавим к имеющемуся датасету data - количество слов в каждом пресс-релизе, решение по ставке

# сократим даты до месяца для data
data['year_month'] = data['date'].dt.to_period('M')

# сокртим даты до месяца для text
text['year_month'] = text['date'].dt.to_period('M')

# добавим количество слов в пресс-релизе в отдельную колонку
text['num_tokens'] = text['text_tokens'].apply(len)

#добавляем столбец со ставкой прошлого месяца
data['key_rate_prev'] = data['key-rate'].shift(-1)

#добавим к data столбец с решением, количеством слов
text_small = text[['year_month', 'decision', 'num_tokens']]
data = data.merge(text_small, on='year_month', how='left', suffixes=('', '_from_text'))

# убираем лишний столбец с индексами
data = data.drop(['Unnamed: 0'], axis=1)

# убираем строки, в которых нет решения 
data = data.dropna(subset=['decision'])

In [11]:
data.reset_index(drop=True, inplace=True)

In [12]:
data.head(10)

,date,key-rate,inflation,dollar_rate,year_month,key_rate_prev,decision,num_tokens
0,2025-09-01,17.0,7.98,80.4261,2025-09,18.0,-1.0,485.0
1,2025-07-01,18.0,8.79,78.5284,2025-07,20.0,-1.0,508.0
2,2025-06-01,20.0,9.40,79.1285,2025-06,21.0,-1.0,467.0
3,2025-04-01,21.0,10.23,85.4963,2025-04,21.0,0.0,486.0
4,2025-03-01,21.0,10.34,88.2568,2025-03,21.0,1.0,539.0
5,2025-02-01,21.0,10.06,97.8107,2025-02,21.0,1.0,536.0
6,2024-12-01,21.0,9.52,107.1758,2024-12,21.0,1.0,542.0
7,2024-10-01,21.0,8.54,93.2221,2024-10,19.0,1.0,475.0
8,2024-09-01,19.0,8.63,90.0013,2024-09,18.0,1.0,523.0
9,2024-07-01,18.0,9.13,87.2972,2024-07,16.0,1.0,458.0


In [13]:
X = data[['inflation', 'dollar_rate', 'num_tokens', 'key_rate_prev']]
y = data['decision']

In [14]:
# разделяем датасет на трейн и тест

from sklearn.model_selection import train_test_split
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.25, random_state=42)

In [16]:
from sklearn.preprocessing import StandardScaler

# делаем объект масштабирования
scaler = StandardScaler()
# считаем параметры стандартизации на трейне
model = scaler.fit(Xtrain)
# стандартизируем признаки для тест и трейн
scaler_data_train = scaler.transform(Xtrain)
scaler_data_test = scaler.transform(Xtest)
# создаем обратно датафрейм со стандартизированными признаками
Xtest_scaled = pd.DataFrame(data=scaler_data_test, columns=Xtest.columns, index=Xtest.index)
Xtrain_scaled = pd.DataFrame(data=scaler_data_train, columns=Xtrain.columns, index=Xtrain.index)

In [17]:
from sklearn.svm import SVC 
from sklearn.metrics import f1_score, accuracy_score

model = SVC(kernel = 'linear')

model.fit(scaler_data_train, ytrain)

pred = model.predict(scaler_data_test)

print("Accuracy:", accuracy_score(ytest, pred))
print("F1 weighted:", f1_score(ytest, pred, average='weighted'))

Accuracy: 0.5769230769230769
F1 weighted: 0.5507692307692308


Модель дала средние результаты, поэтому следует дальше попробовать изучить другие модели.

In [18]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, accuracy_score

# модель KNN
knn = KNeighborsClassifier()

# параметры для подбора
param_grid = {
    'n_neighbors': range(1, 21),  # количество соседей
    'weights': ['uniform', 'distance'],  # равные веса или взвешенные по расстоянию
    'metric': ['euclidean', 'manhattan']  # тип расстояния
}

# GridSearch с 5-кратной кросс-валидацией и F1 (weighted) как метрика
grid_search = GridSearchCV(knn, param_grid, cv=5, scoring='f1_weighted')

# обучаем модель
grid_search.fit(scaler_data_train, ytrain)

# выводим лучшие параметры
print("Лучшие параметры:", grid_search.best_params_)

# выбираем лучшую модель
best_knn = grid_search.best_estimator_

# делаем предсказание на тесте
y_pred = best_knn.predict(scaler_data_test)

# выводим метрики
print("Accuracy:", accuracy_score(ytest, y_pred))
print("F1 weighted:", f1_score(ytest, y_pred, average='weighted'))


Лучшие параметры: {'metric': 'manhattan', 'n_neighbors': 11, 'weights': 'distance'}
Accuracy: 0.8076923076923077
F1 weighted: 0.8095652173913043


In [19]:
y_pred

array([-1., -1., -1., -1.,  1.,  1.,  0., -1.,  1.,  1.,  1.,  1.,  1.,
       -1., -1.,  1., -1.,  1.,  1., -1.,  0., -1.,  1., -1., -1.,  1.])